In [1]:
# Environment setup
import importlib.util
import subprocess
import sys
required = ["numpy", "pandas", "matplotlib"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Environment ready.")

Environment ready.


## Reconstructed implementation - """Faithful Python reconstruction of the supplied MATLAB ES and TSP workflows.

In [2]:
"""Faithful Python reconstruction of the supplied MATLAB ES and TSP workflows.

Designed for execution in Google Colab or locally. The implementation keeps the
original algorithmic choices, while fixing demonstrable execution bugs and adding
portable, reproducible validation/export tooling.
"""

from __future__ import annotations

import argparse
import csv
import json
import math
import os
import platform
import random
import re
import shutil
import sys
import time
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Callable, Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Reconstructed implementation - Reproducibility and runtime utilities

In [3]:
# Reproducibility and runtime utilities

## Reconstructed implementation - def set_global_seed(seed: int) -> None:

In [4]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def runtime_metadata(seed: int) -> dict[str, Any]:
    metadata = {
        "python_version": sys.version,
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "matplotlib_version": plt.matplotlib.__version__,
        "seed": seed,
    }
    try:
        import scipy  # type: ignore
        metadata["scipy_version"] = scipy.__version__
    except Exception:
        metadata["scipy_version"] = None
    try:
        import torch  # type: ignore
        metadata["torch_version"] = torch.__version__
        metadata["cuda_available"] = bool(torch.cuda.is_available())
        if torch.cuda.is_available():
            metadata["gpu"] = torch.cuda.get_device_name(0)
    except Exception:
        metadata["torch_version"] = None
        metadata["cuda_available"] = False
        metadata["gpu"] = None
    return metadata


def is_colab() -> bool:
    try:
        from google.colab import files  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path




WORKFLOW_INVENTORY = [
    {
        "id": "Workflow 1",
        "name": "Evolution Strategy - Main Single Run",
        "source": ["ES_Eshraghi_Sharifi/code/main.m"],
        "description": "Self-adaptive ES with configurable crossover, mutation strategy, survivor selection, and Ackley objective.",
    },
    {
        "id": "Workflow 2",
        "name": "Evolution Strategy - Repeated Runs",
        "source": ["ES_Eshraghi_Sharifi/code/main_iterative.m"],
        "description": "Twenty independent ES runs in the supplied MATLAB implementation, summarized by mean iterations to threshold.",
    },
    {
        "id": "Workflow 3",
        "name": "Traveling Salesman Problem - Main Single Run",
        "source": ["TSP_Eshraghi_Sharifi/code/main.m"],
        "description": "Permutation evolutionary search using OX-1 crossover, swap mutation, tournament selection, and lambda-mu survivor selection.",
    },
    {
        "id": "Workflow 4",
        "name": "Traveling Salesman Problem - Repeated Runs",
        "source": ["TSP_Eshraghi_Sharifi/code/main_iterative.m"],
        "description": "Thirty independent TSP runs driven by data.txt in the supplied MATLAB code; workbook output is replaced by portable per-run CSV/JSON exports.",
    },
]

## Reconstructed implementation - ES objective functions

In [5]:
# ES objective functions

## Reconstructed implementation - def ackley(individual: np.ndarray) -> float:

In [6]:
def ackley(individual: np.ndarray) -> float:
    x = np.asarray(individual, dtype=float).ravel()
    d = x.size
    term1 = -0.2 * np.linalg.norm(x, 2) / math.sqrt(d)
    term2 = np.sum(np.cos(2.0 * math.pi * x)) / d
    return float(-20.0 * math.exp(term1) - math.exp(term2) + 20.0 + math.e)


def rosenbrock_shifted_200(individual: np.ndarray) -> float:
    """Alternative objective visible in the standalone ES costFunction.m archive."""
    x = np.asarray(individual, dtype=float).ravel()
    total = 0.0
    for i in range(x.size - 1):
        total += 100.0 * (x[i + 1] - x[i] ** 2) ** 2 + (x[i] - 1.0) ** 2
    return float(total + 200.0)

## Reconstructed implementation - ES representation and operators

In [7]:
# ES representation and operators

## Reconstructed implementation - @dataclass

In [8]:
@dataclass
class ESPopulationMember:
    variable: np.ndarray
    parameter: Any
    cost: float


def _soft_floor(values: np.ndarray | float, epsilon: float) -> np.ndarray | float:
    """Equivalent to wthresh(x,'s',epsilon)+epsilon for nonnegative x."""
    arr = np.asarray(values, dtype=float)
    out = np.maximum(arr - epsilon, 0.0) + epsilon
    return float(out) if out.ndim == 0 else out


def initialize_es_population(
    rng: np.random.Generator,
    n_population: int,
    var_length: int,
    var_min: float,
    var_max: float,
    method: str,
    objective: Callable[[np.ndarray], float],
) -> list[ESPopulationMember]:
    pop: list[ESPopulationMember] = []
    epsilon = 0.01
    for _ in range(n_population):
        if method == "uncorrelated_1step":
            tau = 1.0 / math.sqrt(var_length)
            sigma_p = rng.lognormal(mean=0.0, sigma=tau**2)
            parameter = float(_soft_floor(sigma_p, epsilon))
        elif method == "uncorrelated_nstep":
            tau = 1.0 / math.sqrt(2.0 * math.sqrt(var_length))
            tau_p = 1.0 / math.sqrt(2.0 * var_length)
            common = rng.lognormal(mean=0.0, sigma=tau**2)
            component = rng.lognormal(mean=0.0, sigma=tau_p**2, size=var_length)
            parameter = _soft_floor(common * component, epsilon)
        elif method == "correlated":
            tau = 1.0 / math.sqrt(2.0 * math.sqrt(var_length))
            tau_p = 1.0 / math.sqrt(2.0 * var_length)
            common = rng.lognormal(mean=0.0, sigma=tau**2)
            component = rng.lognormal(mean=0.0, sigma=tau_p**2, size=var_length)
            sigma = _soft_floor(common * component, epsilon)
            alpha = rng.normal(size=(var_length, var_length))
            alpha = (alpha - alpha.T) / math.sqrt(2.0)
            parameter = {"sigma": np.asarray(sigma), "alpha": alpha}
        else:
            raise ValueError(f"Unknown ES mutation method: {method}")

        variable = rng.uniform(var_min, var_max, size=var_length)
        pop.append(ESPopulationMember(variable=variable, parameter=parameter, cost=objective(variable)))
    return pop


def _copy_parameter(parameter: Any) -> Any:
    if isinstance(parameter, dict):
        return {k: np.array(v, copy=True) for k, v in parameter.items()}
    if isinstance(parameter, np.ndarray):
        return np.array(parameter, copy=True)
    return float(parameter)


def crossover_es(
    rng: np.random.Generator,
    population: list[ESPopulationMember],
    nc: int,
    pc: float,
    mode: str,
    method: str,
) -> list[ESPopulationMember]:
    length_variable = population[0].variable.size
    children: list[ESPopulationMember] = []
    for _ in range(nc):
        p1 = population[int(rng.integers(0, len(population)))]
        p2 = population[int(rng.integers(0, len(population)))]
        c = ESPopulationMember(variable=p1.variable.copy(), parameter=_copy_parameter(p1.parameter), cost=0.0)
        if rng.random() < pc:
            pb1 = rng.integers(0, 2, size=length_variable)
            pb2 = rng.integers(0, 2, size=length_variable)
            if mode == "discrete":
                c.variable = pb1 * p1.variable + (1 - pb1) * p2.variable
                if method == "correlated":
                    temp = rng.integers(0, 2, size=(length_variable, length_variable), dtype=int)
                    pb2_alpha = np.logical_xor(temp.astype(bool), temp.T.astype(bool)).astype(float)
                    c.parameter = {
                        "sigma": pb2 * p1.parameter["sigma"] + (1 - pb2) * p2.parameter["sigma"],
                        "alpha": pb2_alpha * p1.parameter["alpha"] + (1 - pb2_alpha) * p2.parameter["alpha"],
                    }
                else:
                    if method == "uncorrelated_1step":
                        c.parameter = p1.parameter if int(pb2[0]) else p2.parameter
                    else:
                        c.parameter = pb2 * p1.parameter + (1 - pb2) * p2.parameter
            elif mode == "avg":
                c.variable = (p1.variable + p2.variable) / 2.0
                if method == "correlated":
                    c.parameter = {
                        "sigma": (p1.parameter["sigma"] + p2.parameter["sigma"]) / 2.0,
                        "alpha": (p1.parameter["alpha"] + p2.parameter["alpha"]) / 2.0,
                    }
                else:
                    c.parameter = (p1.parameter + p2.parameter) / 2.0
            elif mode == "avg_discrete":
                c.variable = (p1.variable + p2.variable) / 2.0
                if method == "correlated":
                    temp = rng.integers(0, 2, size=(length_variable, length_variable), dtype=int)
                    pb2_alpha = np.logical_xor(temp.astype(bool), temp.T.astype(bool)).astype(float)
                    c.parameter = {
                        "sigma": pb2 * p1.parameter["sigma"] + (1 - pb2) * p2.parameter["sigma"],
                        "alpha": pb2_alpha * p1.parameter["alpha"] + (1 - pb2_alpha) * p2.parameter["alpha"],
                    }
                else:
                    c.parameter = pb2 * p1.parameter + (1 - pb2) * p2.parameter
            elif mode == "discrete_avg":
                c.variable = pb1 * p1.variable + (1 - pb1) * p2.variable
                if method == "correlated":
                    c.parameter = {
                        "sigma": (p1.parameter["sigma"] + p2.parameter["sigma"]) / 2.0,
                        "alpha": (p1.parameter["alpha"] + p2.parameter["alpha"]) / 2.0,
                    }
                else:
                    c.parameter = (p1.parameter + p2.parameter) / 2.0
            else:
                raise ValueError(f"Unknown ES crossover mode: {mode}")
        children.append(c)
    return children


def mutate_es(
    rng: np.random.Generator,
    population: list[ESPopulationMember],
    pm: float,
    mode: str,
    var_min: float,
    var_max: float,
) -> list[ESPopulationMember]:
    length_variable = population[0].variable.size
    result: list[ESPopulationMember] = []
    epsilon = 0.01
    for parent in population:
        c = ESPopulationMember(parent.variable.copy(), _copy_parameter(parent.parameter), parent.cost)
        if rng.random() < pm:
            if mode == "uncorrelated_1step":
                tau = 1.0 / math.sqrt(length_variable)
                sigma_p = rng.lognormal(mean=0.0, sigma=tau**2)
                parent_sigma = float(np.asarray(parent.parameter).reshape(-1)[0])
                c.parameter = float(_soft_floor(sigma_p * parent_sigma, epsilon))
                cov_matrix = c.parameter * np.eye(length_variable)
            elif mode == "uncorrelated_nstep":
                tau = 1.0 / math.sqrt(2.0 * math.sqrt(length_variable))
                tau_p = 1.0 / math.sqrt(2.0 * length_variable)
                common = rng.lognormal(mean=0.0, sigma=tau**2)
                component = rng.lognormal(mean=0.0, sigma=tau_p**2, size=length_variable)
                sigma_p = common * component
                c.parameter = np.asarray(_soft_floor(sigma_p * parent.parameter, epsilon))
                cov_matrix = np.diag(c.parameter)
            elif mode == "correlated":
                beta = 5.0 * math.pi / 180.0
                tau = 1.0 / math.sqrt(2.0 * math.sqrt(length_variable))
                tau_p = 1.0 / math.sqrt(2.0 * length_variable)
                common = rng.lognormal(mean=0.0, sigma=tau**2)
                component = rng.lognormal(mean=0.0, sigma=tau_p**2, size=length_variable)
                sigma_p = common * component
                sigma = np.asarray(_soft_floor(sigma_p * parent.parameter["sigma"], epsilon))
                normal_matrix = rng.normal(size=(length_variable, length_variable))
                alpha_p = beta * (normal_matrix - normal_matrix.T) / math.sqrt(2.0)
                # Preserve the MATLAB implementation's intended wrap-to-pi step.
                alpha_p = (alpha_p + math.pi) % (2.0 * math.pi) - math.pi
                alpha = parent.parameter["alpha"] + alpha_p
                alpha = (alpha + math.pi) % (2.0 * math.pi) - math.pi
                c.parameter = {"sigma": sigma, "alpha": alpha}
                # IMPORTANT: the source code computes diag(sigma) and never applies
                # alpha to a rotated covariance. This exact behavior is preserved.
                cov_matrix = np.diag(sigma)
            else:
                raise ValueError(f"Unknown ES mutation mode: {mode}")

            noise = rng.multivariate_normal(np.zeros(length_variable), cov_matrix)
            c.variable = np.clip(parent.variable + noise, var_min, var_max)
            c.cost = 0.0
        result.append(c)
    return result


def select_es_survivors(
    population: list[ESPopulationMember],
    children: list[ESPopulationMember],
    mode: str,
) -> tuple[list[ESPopulationMember], ESPopulationMember]:
    mu = len(population)
    if mode == "landa+mua":
        combined = population + children
    elif mode == "landa_va_mua":
        combined = children
    else:
        raise ValueError(f"Unknown ES survivor mode: {mode}")
    combined = sorted(combined, key=lambda p: p.cost)
    new_pop = combined[:mu]
    return new_pop, combined[0]


@dataclass
class ESConfig:
    n_population: int = 10
    var_length: int = 5
    iteration: int = 100
    pc: float = 0.3
    pm: float = 0.9
    var_min: float = -30.0
    var_max: float = 30.0
    nc_multiplier: int = 7
    crossover_mode: str = "avg_discrete"
    survivor_mode: str = "landa_va_mua"
    mutation_method: str = "correlated"
    threshold: float = 0.1


def run_es(
    cfg: ESConfig,
    seed: int,
    objective: Callable[[np.ndarray], float] = ackley,
    record_history: bool = True,
) -> dict[str, Any]:
    rng = np.random.default_rng(seed)
    population = initialize_es_population(
        rng, cfg.n_population, cfg.var_length, cfg.var_min, cfg.var_max,
        cfg.mutation_method, objective,
    )
    best_initial = min(population, key=lambda p: p.cost)
    history: list[float] = []
    best_vars: list[np.ndarray] = []
    for iteration in range(1, cfg.iteration + 1):
        children = crossover_es(rng, population, cfg.n_population * cfg.nc_multiplier, cfg.pc, cfg.crossover_mode, cfg.mutation_method)
        children = mutate_es(rng, children, cfg.pm, cfg.mutation_method, cfg.var_min, cfg.var_max)
        for child in children:
            child.cost = objective(child.variable)
        population, best = select_es_survivors(population, children, cfg.survivor_mode)
        history.append(best.cost)
        best_vars.append(best.variable.copy())
        if best.cost <= cfg.threshold:
            break
    return {
        "config": asdict(cfg),
        "seed": seed,
        "objective": getattr(objective, "__name__", str(objective)),
        "iterations_executed": len(history),
        "best_initial_cost": float(best_initial.cost),
        "best_cost": float(history[-1]),
        "best_variable": best_vars[-1].tolist(),
        "history": history if record_history else [],
        "best_variables": np.asarray(best_vars).tolist() if record_history else [],
    }


def run_es_repeated(cfg: ESConfig, count: int, seed: int, objective: Callable[[np.ndarray], float] = ackley) -> dict[str, Any]:
    runs = [run_es(cfg, seed + i, objective=objective) for i in range(count)]
    iterations = np.array([r["iterations_executed"] for r in runs], dtype=float)
    converged = np.array([r["best_cost"] <= cfg.threshold for r in runs], dtype=bool)
    return {
        "config": asdict(cfg),
        "count": count,
        "seed": seed,
        "mean_iterations": float(iterations.mean()),
        "std_iterations": float(iterations.std(ddof=1)) if count > 1 else 0.0,
        "converged_runs": int(converged.sum()),
        "runs": runs,
    }


def describe_workflows() -> list[dict[str, Any]]:
    return WORKFLOW_INVENTORY.copy()


def run_es_workflow1(seed: int = 12345, fast_validation: bool = True) -> dict[str, Any]:
    cfg = ESConfig(
        n_population=10 if fast_validation else 50,
        var_length=5 if fast_validation else 10,
        iteration=100 if fast_validation else 200000,
        pc=0.3, pm=0.9,
        var_min=-30 if fast_validation else -10,
        var_max=30 if fast_validation else 10,
        nc_multiplier=7,
        crossover_mode="avg_discrete",
        survivor_mode="landa_va_mua",
        mutation_method="correlated",
        threshold=0.1,
    )
    return run_es(cfg, seed, objective=ackley)


def run_es_workflow2(seed: int = 12345, count: int = 20, fast_validation: bool = True) -> dict[str, Any]:
    cfg = ESConfig(
        n_population=10, var_length=5, iteration=100 if fast_validation else 100,
        pc=0.3, pm=0.9, var_min=-30, var_max=30, nc_multiplier=7,
        crossover_mode="avg_discrete", survivor_mode="landa_va_mua",
        mutation_method="correlated", threshold=0.1,
    )
    return run_es_repeated(cfg, count=count, seed=seed, objective=ackley)


def run_tsp_workflow3(distance: np.ndarray, seed: int = 12345, fast_validation: bool = True) -> dict[str, Any]:
    cfg = TSPConfig(
        n_population=10 if fast_validation else 50,
        iterations=80 if fast_validation else 10000,
        pc=0.8, pm=0.2,
        n_children=20 if fast_validation else 150,
        crossover_mode="Order_1_crossover", mutation_mode="swap",
        selection_mode="tournoment", survivor_mode="landa_va_mua",
    )
    return run_tsp(distance, cfg, seed)


def run_tsp_workflow4(distance: np.ndarray, seed: int = 12345, count: int = 30, fast_validation: bool = True) -> dict[str, Any]:
    cfg = TSPConfig(
        n_population=10 if fast_validation else 50,
        iterations=80 if fast_validation else 200,
        pc=0.8, pm=0.2,
        n_children=24,
        crossover_mode="Order_1_crossover", mutation_mode="inversion",
        selection_mode="tournoment", survivor_mode="landa_va_mua",
        stop_at_fitness=6.0 if distance.shape[0] == 6 else None,
    )
    runs=[]
    for i in range(count if fast_validation else 1):
        runs.append(run_tsp(distance, cfg, seed+i))
        if runs[-1]["best_cost"] <= 1.0 / 6.0 + 1e-12 and distance.shape[0] == 6:
            break
    iters=np.array([r["iterations_executed"] for r in runs],dtype=float)
    return {
        "workflow": WORKFLOW_INVENTORY[3],
        "config": asdict(cfg),
        "count_executed": len(runs),
        "mean_iterations": float(iters.mean()),
        "runs": runs,
    }

## Reconstructed implementation - TSP parsing, objectives, operators

In [9]:
# TSP parsing, objectives, operators

## Reconstructed implementation - @dataclass

In [10]:
@dataclass
class TSPInstance:
    name: str
    coordinates: np.ndarray
    dimension: int
    edge_weight_type: str = "EUC_2D"


def parse_tsplib(path: str | Path) -> TSPInstance:
    p = Path(path)
    lines = p.read_text(errors="ignore").splitlines()
    name = p.stem
    dimension = None
    edge_weight_type = "EUC_2D"
    coords: list[tuple[float, float]] = []
    in_coords = False
    for line in lines:
        s = line.strip()
        if not s:
            continue
        upper = s.upper()
        if upper.startswith("NAME") and ":" in s:
            name = s.split(":", 1)[1].strip()
        elif upper.startswith("DIMENSION") and ":" in s:
            dimension = int(re.search(r"\d+", s.split(":", 1)[1]).group())
        elif upper.startswith("EDGE_WEIGHT_TYPE") and ":" in s:
            edge_weight_type = s.split(":", 1)[1].strip().upper()
        elif "NODE_COORD_SECTION" in upper:
            in_coords = True
        elif in_coords:
            if upper == "EOF":
                break
            parts = s.split()
            if len(parts) >= 3 and re.fullmatch(r"[+-]?\d+", parts[0]):
                coords.append((float(parts[1]), float(parts[2])))
    coordinates = np.asarray(coords, dtype=float)
    if dimension is None:
        dimension = len(coordinates)
    if len(coordinates) != dimension:
        raise ValueError(f"Parsed {len(coordinates)} coordinates but DIMENSION={dimension} in {p}")
    return TSPInstance(name=name, coordinates=coordinates, dimension=dimension, edge_weight_type=edge_weight_type)


def euclidean_distance_matrix(coordinates: np.ndarray) -> np.ndarray:
    diff = coordinates[:, None, :] - coordinates[None, :, :]
    return np.sqrt(np.sum(diff * diff, axis=2))


def load_distance_matrix_text(path: str | Path) -> np.ndarray:
    """Load the square distance matrix appearing after the 5 header values."""
    vals = np.loadtxt(path, ndmin=1)
    if vals.ndim == 1:
        lines = Path(path).read_text().splitlines()
        numeric_lines = [line for line in lines[5:] if line.strip()]
        rows = [[float(x) for x in line.split()] for line in numeric_lines]
        matrix = np.asarray(rows, dtype=float)
    else:
        matrix = np.asarray(vals, dtype=float)
        if matrix.shape[0] >= 6 and matrix.shape[0] == matrix.shape[1] + 5:
            matrix = matrix[5:, :]
    return matrix


def tsp_cost(routes: np.ndarray, distance: np.ndarray) -> np.ndarray:
    routes = np.asarray(routes, dtype=int)
    out = np.empty(routes.shape[0], dtype=float)
    for i, route in enumerate(routes):
        nxt = np.roll(route, -1)
        edges = distance[route, nxt]
        edges = np.where(edges == -1, np.inf, edges)
        out[i] = np.sum(edges)
    return out


def tsp_fitness(routes: np.ndarray, distance: np.ndarray) -> np.ndarray:
    costs = tsp_cost(routes, distance)
    with np.errstate(divide="ignore", invalid="ignore"):
        return 1.0 / costs


def _random_two_cuts(rng: np.random.Generator, n: int) -> tuple[int, int]:
    a, b = np.sort(rng.choice(n, size=2, replace=False))
    return int(a), int(b)


def pmx(parent1: np.ndarray, parent2: np.ndarray, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    n = len(parent1)
    a, b = _random_two_cuts(rng, n)
    c1 = np.full(n, -1, dtype=int)
    c2 = np.full(n, -1, dtype=int)
    c1[a:b + 1] = parent1[a:b + 1]
    c2[a:b + 1] = parent2[a:b + 1]
    for i in range(a, b + 1):
        if parent2[i] not in c1[a:b + 1]:
            pos = i
            while c1[pos] != -1:
                value = c1[pos]
                pos = int(np.where(parent2 == value)[0][0])
            c1[pos] = parent2[i]
        if parent1[i] not in c2[a:b + 1]:
            pos = i
            while c2[pos] != -1:
                value = c2[pos]
                pos = int(np.where(parent1 == value)[0][0])
            c2[pos] = parent1[i]
    for i in range(n):
        if c1[i] == -1:
            c1[i] = parent2[i]
        if c2[i] == -1:
            c2[i] = parent1[i]
    return c1, c2


def order1(parent1: np.ndarray, parent2: np.ndarray, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    n = len(parent1)
    a, b = _random_two_cuts(rng, n)
    c1 = np.full(n, -1, dtype=int)
    c2 = np.full(n, -1, dtype=int)
    c1[a:b + 1] = parent1[a:b + 1]
    c2[a:b + 1] = parent2[a:b + 1]
    idx = b + 1
    for value in list(parent2[b + 1:]) + list(parent2[:b + 1]):
        if value not in c1:
            if idx >= n:
                idx = 0
            while c1[idx] != -1:
                idx += 1
                if idx >= n:
                    idx = 0
            c1[idx] = value
            idx += 1
    idx = b + 1
    for value in list(parent1[b + 1:]) + list(parent1[:b + 1]):
        if value not in c2:
            if idx >= n:
                idx = 0
            while c2[idx] != -1:
                idx += 1
                if idx >= n:
                    idx = 0
            c2[idx] = value
            idx += 1
    return c1, c2


def cycle_crossover(parent1: np.ndarray, parent2: np.ndarray, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    n = len(parent1)
    c1 = np.full(n, -1, dtype=int)
    c2 = np.full(n, -1, dtype=int)
    visited = np.zeros(n, dtype=bool)
    cycle = 0
    while not visited.all():
        start = int(np.where(~visited)[0][0])
        idxs: list[int] = []
        idx = start
        while not visited[idx]:
            visited[idx] = True
            idxs.append(idx)
            idx = int(np.where(parent1 == parent2[idx])[0][0])
        if cycle % 2 == 0:
            c1[idxs] = parent1[idxs]
            c2[idxs] = parent2[idxs]
        else:
            c1[idxs] = parent2[idxs]
            c2[idxs] = parent1[idxs]
        cycle += 1
    return c1, c2


def mox(parent1: np.ndarray, parent2: np.ndarray, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    n = len(parent1)
    cut = int(rng.integers(1, n))
    head1 = parent1[:cut]
    head2 = parent2[:cut]
    tail2 = [x for x in parent2 if x not in head1]
    tail1 = [x for x in parent1 if x not in head2]
    return np.concatenate([head1, np.asarray(tail2)]), np.concatenate([head2, np.asarray(tail1)])


def tsp_crossover_pair(parent1: np.ndarray, parent2: np.ndarray, mode: str, rng: np.random.Generator, pc: float) -> tuple[np.ndarray, np.ndarray]:
    if rng.random() >= pc:
        return parent1.copy(), parent2.copy()
    mode_map = {
        "pmx": pmx,
        "Order_1_crossover": order1,
        "Cycle_crossover": cycle_crossover,
        "MOX": mox,
    }
    if mode not in mode_map:
        raise ValueError(f"Unknown TSP crossover mode: {mode}")
    return mode_map[mode](parent1, parent2, rng)


def mutate_route(route: np.ndarray, mode: str, rng: np.random.Generator) -> np.ndarray:
    out = route.copy()
    a, b = _random_two_cuts(rng, len(out))
    if mode == "swap":
        out[a], out[b] = out[b], out[a]
    elif mode == "insert":
        value = out[b]
        out = np.concatenate([out[:a], np.array([value]), out[a:b], out[b + 1:]])
    elif mode == "inversion":
        out[a:b + 1] = out[a:b + 1][::-1]
    elif mode == "scramble":
        segment = out[a:b + 1].copy()
        rng.shuffle(segment)
        out[a:b + 1] = segment
    else:
        raise ValueError(f"Unknown TSP mutation mode: {mode}")
    return out


def select_one(
    population: np.ndarray,
    fitness: np.ndarray,
    mode: str,
    rng: np.random.Generator,
) -> np.ndarray:
    n = population.shape[0]
    if n == 0:
        raise ValueError("Cannot select from an empty population")
    if mode == "tournoment":
        k = max(1, int(math.floor(0.5 * n)))
        idx = rng.choice(n, size=k, replace=False)
        winner = idx[np.argmax(fitness[idx])]
        return population[winner].copy()
    if mode == "roletwheel":
        logits = np.asarray(fitness, dtype=float)
        max_logit = np.nanmax(logits)
        weights = np.exp(np.clip(logits - max_logit, -700, 0))
        total = weights.sum()
        if not np.isfinite(total) or total <= 0:
            return population[int(rng.integers(0, n))].copy()
        probs = weights / total
        idx = int(rng.choice(n, p=probs))
        return population[idx].copy()
    if mode == "trancation":
        t = max(1, int(math.floor(0.5 * n)))
        top = np.argsort(-fitness)[:t]
        return population[int(rng.choice(top))].copy()
    if mode == "fullrandom":
        return population[int(rng.integers(0, n))].copy()
    raise ValueError(f"Unknown TSP selection mode: {mode}")


def tsp_crossover(
    population: np.ndarray,
    fitness: np.ndarray,
    nc: int,
    pc: float,
    mode: str,
    mode_select: str,
    rng: np.random.Generator,
) -> np.ndarray:
    if nc % 2 != 0:
        raise ValueError("TSP crossover requires an even number of children, matching MATLAB's nc/2 implementation.")
    children = []
    for _ in range(nc // 2):
        p1 = select_one(population, fitness, mode_select, rng)
        p2 = select_one(population, fitness, mode_select, rng)
        c1, c2 = tsp_crossover_pair(p1, p2, mode, rng, pc)
        children.extend([c1, c2])
    return np.asarray(children, dtype=int)


def tsp_mutation(
    population: np.ndarray,
    fitness: np.ndarray,
    pm: float,
    mode: str,
    mode_select: str,
    rng: np.random.Generator,
) -> np.ndarray:
    result = np.empty_like(population, dtype=int)
    for i in range(len(population)):
        p = select_one(population, fitness, mode_select, rng)
        if rng.random() < pm:
            result[i] = mutate_route(p, mode, rng)
        else:
            result[i] = p
    return result


def tsp_select_remainings(
    population: np.ndarray,
    children: np.ndarray,
    mode: str,
    mode_select: str,
    distance: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    mu = len(population)
    lam = len(children)
    if mode == "landa+mua" or lam < mu:
        pool = np.vstack([population, children])
    elif mode == "landa_va_mua":
        pool = children
    else:
        raise ValueError(f"Unknown survivor mode: {mode}")
    fitness = tsp_fitness(pool, distance)
    new_pop = np.vstack([select_one(pool, fitness, mode_select, rng) for _ in range(mu)])
    return new_pop.astype(int)


@dataclass
class TSPConfig:
    n_population: int = 50
    iterations: int = 1000
    pc: float = 0.8
    pm: float = 0.2
    n_children: int = 150
    crossover_mode: str = "Order_1_crossover"
    mutation_mode: str = "swap"
    selection_mode: str = "tournoment"
    survivor_mode: str = "landa_va_mua"
    stop_at_fitness: Optional[float] = None


def run_tsp(
    distance: np.ndarray,
    cfg: TSPConfig,
    seed: int,
    record_history: bool = True,
) -> dict[str, Any]:
    rng = np.random.default_rng(seed)
    n = distance.shape[0]
    population = np.vstack([rng.permutation(n) for _ in range(cfg.n_population)])
    fitness = tsp_fitness(population, distance)
    initial_idx = int(np.argmax(fitness))
    initial_route = population[initial_idx].copy()
    history: list[float] = []
    history_cost: list[float] = []
    history_best_route: list[list[int]] = []
    start = time.time()
    for iteration in range(1, cfg.iterations + 1):
        children = tsp_crossover(population, fitness, cfg.n_children, cfg.pc, cfg.crossover_mode, cfg.selection_mode, rng)
        child_fitness = tsp_fitness(children, distance)
        children = tsp_mutation(children, child_fitness, cfg.pm, cfg.mutation_mode, cfg.selection_mode, rng)
        population = tsp_select_remainings(population, children, cfg.survivor_mode, cfg.selection_mode, distance, rng)
        fitness = tsp_fitness(population, distance)
        idx = int(np.argmax(fitness))
        best_fit = float(fitness[idx])
        best_cost = float(1.0 / best_fit) if best_fit > 0 else math.inf
        history.append(best_fit)
        history_cost.append(best_cost)
        history_best_route.append(population[idx].tolist())
        if cfg.stop_at_fitness is not None and best_fit >= cfg.stop_at_fitness:
            break
    best_idx = int(np.argmax(fitness))
    elapsed = time.time() - start
    return {
        "config": asdict(cfg),
        "seed": seed,
        "dimension": n,
        "initial_route": initial_route.tolist(),
        "iterations_executed": len(history),
        "best_fitness": float(fitness[best_idx]),
        "best_cost": float(1.0 / fitness[best_idx]),
        "best_route": population[best_idx].tolist(),
        "history_fitness": history if record_history else [],
        "history_cost": history_cost if record_history else [],
        "history_best_route": history_best_route if record_history else [],
        "elapsed_seconds": elapsed,
    }


def validate_route(route: Iterable[int], n: int) -> dict[str, Any]:
    r = np.asarray(list(route), dtype=int)
    counts = np.bincount(r, minlength=n) if r.size else np.array([])
    return {
        "length": int(r.size),
        "expected_length": n,
        "contains_all_city_ids_0_to_n_minus_1": bool(r.size == n and np.array_equal(np.sort(r), np.arange(n))),
        "duplicate_count": int(np.sum(np.maximum(counts - 1, 0))) if r.size else 0,
    }

## Reconstructed implementation - Validation and artifact collection

In [11]:
# Validation and artifact collection

## Reconstructed implementation - def brute_force_tsp_optimum(distance: np.ndarray) -> tuple[float, np.ndarray]:

In [12]:
def brute_force_tsp_optimum(distance: np.ndarray) -> tuple[float, np.ndarray]:
    """Exact optimizer for small validation instances only."""
    import itertools
    n = distance.shape[0]
    if n > 10:
        raise ValueError("Brute force validation is limited to n <= 10")
    best = math.inf
    best_route = None
    # Fix city 0 to remove rotational duplicates.
    for tail in itertools.permutations(range(1, n)):
        route = np.array((0,) + tail, dtype=int)
        cost = float(tsp_cost(route[None, :], distance)[0])
        if cost < best:
            best = cost
            best_route = route.copy()
    return best, best_route  # type: ignore[return-value]


def _find_required_file(base_dir: Path, name: str) -> Path:
    matches = list(base_dir.rglob(name))
    if not matches:
        raise FileNotFoundError(f"Could not locate {name} under {base_dir}")
    return matches[0]


def validate_supplied_tsp_outputs(base_dir: Path) -> list[dict[str, Any]]:
    city1 = _find_required_file(base_dir, "city1.tsp")
    city2 = _find_required_file(base_dir, "city2.tsp")
    city3 = _find_required_file(base_dir, "city3.tsp")
    zi929 = _find_required_file(base_dir, "zi929.txt")
    ar9152 = _find_required_file(base_dir, "ar9152.txt")
    cases = [
        (city1, _find_required_file(base_dir, "myFile_W29.csv")),
        (city2, _find_required_file(base_dir, "myFile1_W38.csv")),
        (city3, _find_required_file(base_dir, "myFile1_W194.csv")),
        (zi929, _find_required_file(base_dir, "myfile929.csv")),
        (ar9152, _find_required_file(base_dir, "myFile9152.csv")),
    ]
    rows = []
    for tsp_path, out_path in cases:
        instance = parse_tsplib(tsp_path)
        text = out_path.read_text().splitlines()
        route = np.array([int(x) for x in text[0].split(",") if x.strip()], dtype=int)
        saved_cost = float([x for x in text[1].split(",") if x.strip()][0])
        distance = euclidean_distance_matrix(instance.coordinates)
        computed = float(tsp_cost(route[None, :], distance)[0])
        rows.append({
            "instance": instance.name,
            "dimension": instance.dimension,
            "output_file": out_path.name,
            "route_valid": validate_route(route, instance.dimension)["contains_all_city_ids_0_to_n_minus_1"],
            "computed_raw_euclidean_cost": computed,
            "stored_cost": saved_cost,
            "absolute_difference": abs(computed - saved_cost),
            "relative_difference": abs(computed - saved_cost) / max(1.0, abs(saved_cost)),
        })
    return rows


def save_json(path: Path, data: Any) -> None:
    path.write_text(json.dumps(data, indent=2, allow_nan=False), encoding="utf-8")


def save_es_result(result: dict[str, Any], workflow_dir: Path, label: str) -> list[Path]:
    ensure_dir(workflow_dir)
    files: list[Path] = []
    summary = {k: v for k, v in result.items() if k not in {"history", "best_variables"}}
    p = workflow_dir / f"{label}_summary.json"
    save_json(p, summary)
    files.append(p)
    if result.get("history"):
        hist = pd.DataFrame({"iteration": np.arange(1, len(result["history"]) + 1), "best_cost": result["history"]})
        hp = workflow_dir / f"{label}_history.csv"
        hist.to_csv(hp, index=False)
        files.append(hp)
        fig = workflow_dir / f"{label}_convergence.png"
        plt.figure(figsize=(8, 4.5))
        plt.plot(hist["iteration"], hist["best_cost"])
        plt.xlabel("Iteration")
        plt.ylabel("Best objective")
        plt.title(f"ES convergence - {label}")
        plt.tight_layout()
        plt.savefig(fig, dpi=160)
        plt.close()
        files.append(fig)
    return files


def save_tsp_result(result: dict[str, Any], workflow_dir: Path, label: str, distance: np.ndarray) -> list[Path]:
    ensure_dir(workflow_dir)
    files: list[Path] = []
    route = np.asarray(result["best_route"], dtype=int)
    route_path = workflow_dir / f"{label}_best_route.csv"
    pd.DataFrame([route]).to_csv(route_path, header=False, index=False)
    files.append(route_path)
    summary = {k: v for k, v in result.items() if k not in {"history_fitness", "history_cost", "history_best_route"}}
    summary["route_validation"] = validate_route(route, len(route))
    save_path = workflow_dir / f"{label}_summary.json"
    save_json(save_path, summary)
    files.append(save_path)
    if result.get("history_cost"):
        hist = pd.DataFrame({"iteration": np.arange(1, len(result["history_cost"]) + 1), "best_cost": result["history_cost"]})
        hp = workflow_dir / f"{label}_history.csv"
        hist.to_csv(hp, index=False)
        files.append(hp)
        fig = workflow_dir / f"{label}_convergence.png"
        plt.figure(figsize=(8, 4.5))
        plt.plot(hist["iteration"], hist["best_cost"])
        plt.xlabel("Iteration")
        plt.ylabel("Best tour length")
        plt.title(f"TSP convergence - {label}")
        plt.tight_layout()
        plt.savefig(fig, dpi=160)
        plt.close()
        files.append(fig)
    return files


def collect_and_zip_results(
    results_root: Path,
    source_root: Path,
    manifest_extra: Optional[dict[str, Any]] = None,
) -> tuple[Path, Path, list[dict[str, Any]]]:
    ensure_dir(results_root)
    summary_dir = ensure_dir(results_root / "summary")
    manifest: list[dict[str, Any]] = []
    if manifest_extra:
        save_json(summary_dir / "run_summary.json", manifest_extra)
    for path in sorted(results_root.rglob("*")):
        if not path.is_file() or path.name == "results_manifest.json":
            continue
        rel = path.relative_to(results_root)
        artifact_type = path.suffix.lower().lstrip(".") or "file"
        if path.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}:
            artifact_type = "image"
        elif path.suffix.lower() in {".csv", ".xlsx"}:
            artifact_type = "table"
        elif path.suffix.lower() in {".json"}:
            artifact_type = "json"
        manifest.append({
            "workflow_name": rel.parts[0] if rel.parts else "summary",
            "artifact_name": path.name,
            "artifact_type": artifact_type,
            "file_path": str(rel),
            "file_size_bytes": path.stat().st_size,
            "creation_status": "created",
        })
    manifest_path = results_root / "results_manifest.json"
    save_json(manifest_path, {"artifacts": manifest, "source_root": str(source_root)})
    manifest.append({
        "workflow_name": "summary",
        "artifact_name": "results_manifest.json",
        "artifact_type": "json",
        "file_path": "results_manifest.json",
        "file_size_bytes": manifest_path.stat().st_size,
        "creation_status": "created",
    })
    zip_path = results_root.parent / "MATLAB_Reconstruction_All_Workflow_Results.zip"
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=results_root.parent, base_dir=results_root.name)
    return zip_path, manifest_path, manifest


def maybe_colab_download(path: Path) -> bool:
    if not is_colab():
        return False
    from google.colab import files  # type: ignore
    files.download(str(path))
    return True


def make_validation_results(base_dir: Path, results_root: Path, seed: int) -> dict[str, Any]:
    ensure_dir(results_root)
    validation: dict[str, Any] = {"seed": seed, "checks": []}

    # ES validation across all three implemented mutation families.
    es_dir = ensure_dir(results_root / "workflow_01_es_validation")
    es_runs = {}
    for i, method in enumerate(["uncorrelated_1step", "uncorrelated_nstep", "correlated"]):
        cfg = ESConfig(
            n_population=10,
            var_length=5,
            iteration=30,
            pc=0.3,
            pm=0.9,
            var_min=-30,
            var_max=30,
            nc_multiplier=7,
            crossover_mode="avg_discrete",
            survivor_mode="landa_va_mua",
            mutation_method=method,
            threshold=0.1,
        )
        result = run_es(cfg, seed + i, objective=ackley)
        save_es_result(result, es_dir, f"ackley_{method}")
        es_runs[method] = {
            "iterations_executed": result["iterations_executed"],
            "best_cost": result["best_cost"],
            "finite": bool(np.isfinite(result["best_cost"])),
            "variable_length": len(result["best_variable"]) == 5,
        }
    validation["checks"].append({"workflow": "ES mutation-family smoke validation", "status": "VALIDATED", "runs": es_runs})

    # Explicit workflow 1 and 2 smoke executions using the supplied main/main_iterative settings.
    w1 = run_es_workflow1(seed=seed + 20, fast_validation=True)
    save_es_result(w1, results_root / "workflow_01_es_main", "main_smoke")
    w2 = run_es_workflow2(seed=seed + 21, count=20, fast_validation=True)
    ensure_dir(results_root / "workflow_02_es_repeated")
    save_json(results_root / "workflow_02_es_repeated" / "repeated_summary.json", w2)
    validation["checks"].append({
        "workflow": "Workflow 1 - ES main",
        "status": "VALIDATED" if np.isfinite(w1["best_cost"]) and len(w1["best_variable"]) == 5 else "PARTIALLY VALIDATED",
        "iterations_executed": w1["iterations_executed"],
        "best_cost": w1["best_cost"],
    })
    validation["checks"].append({
        "workflow": "Workflow 2 - ES repeated",
        "status": "VALIDATED" if w2["count"] == 20 and np.isfinite(w2["mean_iterations"]) else "PARTIALLY VALIDATED",
        "count": w2["count"],
        "mean_iterations": w2["mean_iterations"],
    })

    # Small exact TSP validation.
    tsp6 = np.array([
        [0, 1, 10, 10, 10, 1],
        [1, 0, 1, 10, 10, 10],
        [10, 1, 0, 1, 10, 10],
        [10, 10, 1, 0, 1, 10],
        [10, 10, 10, 1, 0, 1],
        [1, 10, 10, 10, 1, 0],
    ], dtype=float)
    exact_cost, _ = brute_force_tsp_optimum(tsp6)
    tsp_dir = ensure_dir(results_root / "workflow_02_tsp_validation")
    cfg = TSPConfig(
        n_population=10, iterations=80, pc=0.8, pm=0.2, n_children=20,
        crossover_mode="Order_1_crossover", mutation_mode="swap",
        selection_mode="tournoment", survivor_mode="landa_va_mua",
    )
    tsp_result = run_tsp(tsp6, cfg, seed + 10)
    save_tsp_result(tsp_result, tsp_dir, "six_city_demo", tsp6)
    valid = validate_route(tsp_result["best_route"], 6)
    validation["checks"].append({
        "workflow": "TSP six-city exact-consistency validation",
        "status": "VALIDATED" if valid["contains_all_city_ids_0_to_n_minus_1"] and np.isfinite(tsp_result["best_cost"]) else "PARTIALLY VALIDATED",
        "exact_optimum_cost": exact_cost,
        "observed_best_cost": tsp_result["best_cost"],
        "route_validation": valid,
    })

    # Explicit workflow 3 and 4 smoke executions using the supplied main/main_iterative settings.
    w3 = run_tsp_workflow3(tsp6, seed=seed + 30, fast_validation=True)
    save_tsp_result(w3, results_root / "workflow_03_tsp_main", "main_smoke", tsp6)
    w4 = run_tsp_workflow4(tsp6, seed=seed + 31, count=30, fast_validation=True)
    ensure_dir(results_root / "workflow_04_tsp_repeated")
    save_json(results_root / "workflow_04_tsp_repeated" / "repeated_summary.json", w4)
    validation["checks"].append({
        "workflow": "Workflow 3 - TSP main",
        "status": "VALIDATED" if validate_route(w3["best_route"], 6)["contains_all_city_ids_0_to_n_minus_1"] else "PARTIALLY VALIDATED",
        "best_cost": w3["best_cost"],
    })
    validation["checks"].append({
        "workflow": "Workflow 4 - TSP repeated",
        "status": "VALIDATED" if w4["count_executed"] == 30 else "PARTIALLY VALIDATED",
        "count_executed": w4["count_executed"],
        "mean_iterations": w4["mean_iterations"],
    })

    # Verify supplied result files against the code's distance convention when the
    # original project files are available in the execution directory. In Colab,
    # the notebook still runs end-to-end without hidden local inputs.
    try:
        supplied = validate_supplied_tsp_outputs(base_dir)
        validation["supplied_tsp_output_checks"] = supplied
        max_rel = max(r["relative_difference"] for r in supplied)
        validation["checks"].append({
            "workflow": "Supplied TSP output-file forensic validation",
            "status": "VALIDATED" if max_rel < 0.005 else "PARTIALLY VALIDATED",
            "max_relative_difference": max_rel,
            "cases": supplied,
        })
    except FileNotFoundError as exc:
        validation["checks"].append({
            "workflow": "Supplied TSP output-file forensic validation",
            "status": "NOT VALIDATED",
            "reason": str(exc),
            "note": "The source archives were available during reconstruction but are not assumed to be hidden inputs in the standalone Colab notebook.",
        })

    validation_path = results_root / "summary" / "validation_summary.json"
    ensure_dir(validation_path.parent)
    save_json(validation_path, validation)
    return validation


def run_workflows(
    base_dir: Path,
    output_root: Path,
    seed: int,
    mode: str = "validation",
) -> dict[str, Any]:
    ensure_dir(output_root)
    set_global_seed(seed)
    meta = runtime_metadata(seed)
    ensure_dir(output_root / "summary")
    save_json(output_root / "summary" / "runtime_metadata.json", meta)
    save_json(output_root / "summary" / "workflow_inventory.json", WORKFLOW_INVENTORY)

    validation = make_validation_results(base_dir, output_root, seed)

    # Optional full original-scale configurations are exposed here but not run by default.
    full_configs = {
        "ES_main_original": asdict(ESConfig(
            n_population=50, var_length=10, iteration=200000, pc=0.3, pm=0.9,
            var_min=-10, var_max=10, nc_multiplier=7, crossover_mode="avg_discrete",
            survivor_mode="landa_va_mua", mutation_method="correlated", threshold=0.1,
        )),
        "TSP_main_original": asdict(TSPConfig(
            n_population=50, iterations=10000, pc=0.8, pm=0.2, n_children=150,
            crossover_mode="Order_1_crossover", mutation_mode="swap",
            selection_mode="tournoment", survivor_mode="landa_va_mua",
        )),
    }
    save_json(output_root / "summary" / "original_configurations.json", full_configs)
    return {"runtime": meta, "validation": validation, "mode": mode}

## Configuration and input preparation

The six-city matrix below is the exact matrix from `data.txt`. It is used for deterministic, end-to-end validation and for the repeated TSP smoke workflow.

In [13]:
BASE_DIR = Path(".")
OUTPUT_ROOT = Path("results")
SEED = 12345
six_city_distance = np.array([[0,1,10,10,10,1],[1,0,1,10,10,10],[10,1,0,1,10,10],[10,10,1,0,1,10],[10,10,10,1,0,1],[1,10,10,10,1,0]], dtype=float)
print("BASE_DIR:", BASE_DIR.resolve())
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())
print("Seed:", SEED)

BASE_DIR: /content
OUTPUT_ROOT: /content/results
Seed: 12345


## Workflow 1 - Evolution Strategy / Main Single Run

In [14]:
workflow1_result = run_es_workflow1(seed=SEED + 20, fast_validation=True)
print({"iterations_executed": workflow1_result["iterations_executed"], "best_cost": workflow1_result["best_cost"], "best_variable_length": len(workflow1_result["best_variable"])})

{'iterations_executed': 91, 'best_cost': 0.08594946081969512, 'best_variable_length': 5}


## Workflow 2 - Evolution Strategy / Repeated Runs

In [15]:
workflow2_result = run_es_workflow2(seed=SEED + 21, count=20, fast_validation=True)
print({"count": workflow2_result["count"], "mean_iterations": workflow2_result["mean_iterations"], "converged_runs": workflow2_result["converged_runs"]})

{'count': 20, 'mean_iterations': 69.25, 'converged_runs': 19}


## Workflow 3 - TSP / Main Single Run

In [16]:
workflow3_result = run_tsp_workflow3(six_city_distance, seed=SEED + 30, fast_validation=True)
print({"best_cost": workflow3_result["best_cost"], "best_route": workflow3_result["best_route"], "route_validation": validate_route(workflow3_result["best_route"], 6)})

{'best_cost': 6.0, 'best_route': [0, 1, 2, 3, 4, 5], 'route_validation': {'length': 6, 'expected_length': 6, 'contains_all_city_ids_0_to_n_minus_1': True, 'duplicate_count': 0}}


## Workflow 4 - TSP / Repeated Runs

In [17]:
workflow4_result = run_tsp_workflow4(six_city_distance, seed=SEED + 31, count=30, fast_validation=True)
print({"count_executed": workflow4_result["count_executed"], "mean_iterations": workflow4_result["mean_iterations"]})

{'count_executed': 30, 'mean_iterations': 80.0}


## Final automatic result export

**Last executable cell.** It runs the complete bounded validation pipeline, generates a machine-readable manifest, creates the comprehensive results ZIP, prints every exported artifact, and triggers the Google Colab download API when the notebook is running in Colab.

In [18]:
summary = run_workflows(BASE_DIR, OUTPUT_ROOT, SEED, mode="validation")
zip_path, manifest_path, manifest = collect_and_zip_results(OUTPUT_ROOT, BASE_DIR, manifest_extra=summary)
print("\nFINAL EXPORT")
print("Manifest:", manifest_path)
print("ZIP:", zip_path)
print("Exported artifacts:", len(manifest))
for entry in manifest:
    print(f" - {entry['workflow_name']}/{entry['artifact_name']}")
if is_colab():
    maybe_colab_download(zip_path)
    print("Google Colab download initiated.")
else:
    print("Not running inside Google Colab; ZIP created at the path above.")


FINAL EXPORT
Manifest: results/results_manifest.json
ZIP: MATLAB_Reconstruction_All_Workflow_Results.zip
Exported artifacts: 28
 - summary/original_configurations.json
 - summary/run_summary.json
 - summary/runtime_metadata.json
 - summary/validation_summary.json
 - summary/workflow_inventory.json
 - workflow_01_es_main/main_smoke_convergence.png
 - workflow_01_es_main/main_smoke_history.csv
 - workflow_01_es_main/main_smoke_summary.json
 - workflow_01_es_validation/ackley_correlated_convergence.png
 - workflow_01_es_validation/ackley_correlated_history.csv
 - workflow_01_es_validation/ackley_correlated_summary.json
 - workflow_01_es_validation/ackley_uncorrelated_1step_convergence.png
 - workflow_01_es_validation/ackley_uncorrelated_1step_history.csv
 - workflow_01_es_validation/ackley_uncorrelated_1step_summary.json
 - workflow_01_es_validation/ackley_uncorrelated_nstep_convergence.png
 - workflow_01_es_validation/ackley_uncorrelated_nstep_history.csv
 - workflow_01_es_validation/ac

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Google Colab download initiated.
